<a href="https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

In [ ]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [ ]:
REL_APR = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

In [ ]:
%pip -q install -U duckdb huggingface_hub

import duckdb
from google.colab import userdata
from huggingface_hub import whoami

token = userdata.get("HF_TOKEN")

print("Token found:", token is not None)
print("Logged in as:", whoami(token=token)["name"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 25.5 MB/s eta 0:00:00
Token found: True
Logged in as: srijan317


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1) One row represents performance of a specific piece of content for one client on one specific day.
2) I'm using fact_content_daily_performance table.
3) I'm analyzing data from from March 1, 2026 through March 31, 2026.
4) I'm trying to predict which pages will decline in search performance.
5)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample = con.sql(f"""
    SELECT *
    FROM {REL}
    LIMIT 10
""").df()
sample

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,True,False,True,<NA>,239,1,1756,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,True,False,True,<NA>,191,0,1496,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,True,False,True,<NA>,55,0,180,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,True,False,True,<NA>,77,0,434,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,True,False,True,<NA>,2,0,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

1) Feature: historical GSC, GA4, traffic-source, and engagement metrics
2) Context: IDs and dates
3) Excluded: ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude,ai_meta, ai_other(already represented by total sessions_ai)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

All five selected features are historical metrics aggregated from March 2026. Because they were recorded before the prediction period (April), they would already be available when deciding which content is likely to decline. No future information is included in the honest feature set.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#Checks if there are duplicates
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {REL}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [ ]:
#Count+Window
#Makes sures only data from March 1 to March 31 2026 is being analysed
data_count = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {REL}
""").df()

data_count

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
#Availability

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS available_rows
    FROM {REL}
    WHERE
        gsc_data_available IS TRUE
        AND ga4_data_available IS TRUE
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,364347


In [ ]:

data_limits = con.sql(f"""
    SELECT
         client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
FROM {REL}
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()
data_limits

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,client_65de48885f4ef01b,content_5c80451459c29b4a,5.0,0.0,5.400000,1.0,0.0
1,client_65de48885f4ef01b,content_6b0149a80607dac3,1199.0,12.0,8.119012,26.0,3.0
2,client_65de48885f4ef01b,content_62673eea26c31c17,57145.0,43.0,6.814939,125.0,6.0
3,client_65de48885f4ef01b,content_872342e050545a12,39.0,0.0,6.538462,1.0,0.0
4,client_65de48885f4ef01b,content_4c185d1c173cd53d,278.0,9.0,10.399210,16.0,1.0
...,...,...,...,...,...,...,...
63851,client_20259bd6705d81d4,content_d41e42e293236000,7.0,0.0,24.428571,1.0,0.0
63852,client_20259bd6705d81d4,content_bef0c2e83e54fe21,45.0,0.0,24.400000,1.0,0.0
63853,client_20259bd6705d81d4,content_5e0db3274a5b6e28,117.0,1.0,7.222222,1.0,0.0
63854,client_20259bd6705d81d4,content_70db3d9a8ed58d6f,901.0,1.0,5.957825,2.0,0.0


In [ ]:
labeled = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_gsc_impressions
    FROM {REL_APR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.*,
    a.april_gsc_impressions,

    CASE
        WHEN a.april_gsc_impressions < m.gsc_impressions THEN 1
        ELSE 0
    END AS is_declining_label

FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
""").df()

labeled.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,april_gsc_impressions,is_declining_label
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,458.0,2.0,4.418032,14.0,0.0,937.0,0
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3943.0,23.0,4.392897,54.0,1.0,3616.0,1
2,client_65de48885f4ef01b,content_3c286ded8bd68120,2180.0,15.0,8.439390,30.0,2.0,1437.0,1
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,503.0,8.0,5.531459,23.0,1.0,376.0,1
4,client_65de48885f4ef01b,content_ff867882e604fa96,24.0,0.0,2.850000,2.0,0.0,126.0,0


In [ ]:
#Decision Tree Classifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X = labeled[features]
y = labeled["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create and train model
model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9392199090691553


In [ ]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Probability that each page is declining (class 1)
tree_test_score = model.predict_proba(X_test)[:, 1]

# Precision@50
p50 = precision_at_k(tree_test_score, y_test, 50)

print(f"Tree test Precision@50: {p50:.3f}")

Tree test Precision@50: 0.420


In [ ]:
#Leakage Experiment
leaky_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "april_gsc_impressions"
]

X_leaky = labeled[leaky_features]
y_leaky = labeled["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.2,
    random_state=42,
    stratify=y_leaky
)

# Create and train model
model_leaky = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model_leaky.fit(X_train, y_train)

# Prediction
y_pred = model_leaky.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9443247985961554


In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Probability that each page is declining (class 1)
tree_test_score = model_leaky.predict_proba(X_test)[:, 1]

# Precision@50
p50 = precision_at_k(tree_test_score, y_test, 50)

print(f"Tree test Precision@50: {p50:.3f}")

Tree test Precision@50: 0.920


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Not all rows have both GSC and GA4 data available. In the March 2026 slice, only 364,347 of 9,841,378 daily rows had both data sources available. Therefore, the feature analysis using both GSC and GA4 metrics represents only the subset of content with both sources available and may not represent all content in the warehouse.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.